# 07 — The LLM Selection Framework — Practical Examples

**Covers**: TaskRequirements profile, hard constraint filtering, task-fit scoring, monthly cost calculation, automated selection engine, full scorecard

In [ ]:
import os, json, time
from openai import OpenAI
from dataclasses import dataclass, field
from typing import Literal, Optional
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
print('✅ Setup complete')

## 1. Model Registry — Centralized Spec Database

In [ ]:
MODEL_REGISTRY = {
    'gpt-4o': {
        'provider': 'OpenAI',    'context_k': 128, 'on_premise': False, 'multimodal': True,
        'input_per_1m': 5.00,    'output_per_1m': 15.00, 'median_latency_ms': 2000,
        'typical_cost_per_call': 0.012,  # 2k input + 400 output tokens
        'reasoning': 88, 'instruction': 92, 'coding': 85, 'speed': 70, 'cost': 30
    },
    'gpt-4o-mini': {
        'provider': 'OpenAI',    'context_k': 128, 'on_premise': False, 'multimodal': True,
        'input_per_1m': 0.15,    'output_per_1m': 0.60, 'median_latency_ms': 1200,
        'typical_cost_per_call': 0.0005,
        'reasoning': 76, 'instruction': 88, 'coding': 78, 'speed': 82, 'cost': 95
    },
    'claude-3-5-sonnet': {
        'provider': 'Anthropic', 'context_k': 200, 'on_premise': False, 'multimodal': True,
        'input_per_1m': 3.00,    'output_per_1m': 15.00, 'median_latency_ms': 3000,
        'typical_cost_per_call': 0.012,
        'reasoning': 87, 'instruction': 94, 'coding': 92, 'speed': 65, 'cost': 45
    },
    'claude-3-5-haiku': {
        'provider': 'Anthropic', 'context_k': 200, 'on_premise': False, 'multimodal': True,
        'input_per_1m': 0.80,    'output_per_1m': 4.00, 'median_latency_ms': 800,
        'typical_cost_per_call': 0.003,
        'reasoning': 74, 'instruction': 88, 'coding': 76, 'speed': 88, 'cost': 88
    },
    'gemini-1.5-flash': {
        'provider': 'Google',    'context_k': 1000, 'on_premise': False, 'multimodal': True,
        'input_per_1m': 0.075,   'output_per_1m': 0.30, 'median_latency_ms': 700,
        'typical_cost_per_call': 0.0003,
        'reasoning': 72, 'instruction': 80, 'coding': 72, 'speed': 90, 'cost': 98
    },
    'llama-3.1-70b': {
        'provider': 'Meta/OSS',  'context_k': 128, 'on_premise': True, 'multimodal': False,
        'input_per_1m': 0.52,    'output_per_1m': 0.75, 'median_latency_ms': 1500,
        'typical_cost_per_call': 0.001,
        'reasoning': 80, 'instruction': 84, 'coding': 78, 'speed': 75, 'cost': 80
    },
}
print(f'Registry has {len(MODEL_REGISTRY)} models')

## 2. TaskRequirements + Hard Constraint Filtering

In [ ]:
@dataclass
class TaskRequirements:
    task_category: str
    complexity: str
    max_latency_ms: Optional[int] = None
    max_cost_per_call: Optional[float] = None
    min_context_tokens: int = 4_096
    requires_multimodal: bool = False
    privacy_required: bool = False
    calls_per_day: int = 1_000
    min_quality_score: float = 0.80

HARD_CONSTRAINTS = [
    ('Privacy',    lambda r, m: not r.privacy_required or m['on_premise']),
    ('Context',    lambda r, m: m['context_k'] * 1000 >= r.min_context_tokens),
    ('Multimodal', lambda r, m: not r.requires_multimodal or m['multimodal']),
    ('Cost',       lambda r, m: r.max_cost_per_call is None or m['typical_cost_per_call'] <= r.max_cost_per_call),
    ('Latency',    lambda r, m: r.max_latency_ms is None or m['median_latency_ms'] <= r.max_latency_ms),
]

def filter_models(req: TaskRequirements) -> list[str]:
    passed = []
    print('Hard constraint filtering:')
    for name, spec in MODEL_REGISTRY.items():
        results = {cname: check(req, spec) for cname, check in HARD_CONSTRAINTS}
        all_pass = all(results.values())
        failed = [c for c, p in results.items() if not p]
        if all_pass:
            passed.append(name)
            print(f'  ✅ {name:<22} — all constraints pass')
        else:
            print(f'  ❌ {name:<22} — failed: {failed}')
    return passed

# Test: customer support chatbot
REQ = TaskRequirements(
    task_category='chat', complexity='moderate',
    max_latency_ms=3_000, max_cost_per_call=0.005,
    calls_per_day=10_000
)
candidates = filter_models(REQ)
print(f'\n{len(candidates)} models pass hard constraints: {candidates}')

## 3. Task-Fit Scoring — Rank Candidates

In [ ]:
TASK_WEIGHTS = {
    'reasoning':    {'reasoning': 0.40, 'instruction': 0.30, 'coding': 0.10, 'speed': 0.10, 'cost': 0.10},
    'coding':       {'reasoning': 0.20, 'instruction': 0.20, 'coding': 0.40, 'speed': 0.10, 'cost': 0.10},
    'extraction':   {'reasoning': 0.10, 'instruction': 0.40, 'coding': 0.10, 'speed': 0.10, 'cost': 0.30},
    'summarization':{'reasoning': 0.10, 'instruction': 0.30, 'coding': 0.00, 'speed': 0.10, 'cost': 0.50},
    'classification':{'reasoning':0.10, 'instruction': 0.30, 'coding': 0.00, 'speed': 0.20, 'cost': 0.40},
    'chat':         {'reasoning': 0.20, 'instruction': 0.30, 'coding': 0.10, 'speed': 0.30, 'cost': 0.10},
    'qa':           {'reasoning': 0.35, 'instruction': 0.30, 'coding': 0.10, 'speed': 0.10, 'cost': 0.15},
}

def score_candidates(task: str, candidates: list[str]) -> list[tuple[str, float]]:
    weights = TASK_WEIGHTS.get(task, TASK_WEIGHTS['chat'])
    scores = []
    for model in candidates:
        spec = MODEL_REGISTRY[model]
        score = sum(spec.get(cap, 50) * w for cap, w in weights.items())
        scores.append((model, score))
    return sorted(scores, key=lambda x: x[1], reverse=True)

ranked = score_candidates(REQ.task_category, candidates)
print(f'Task-fit scores for [{REQ.task_category}]:')
print(f"{'Model':<22} {'Score':>8} {'Recommendation'}")
print('-' * 50)
for i, (m, s) in enumerate(ranked):
    rec = '⭐ RECOMMENDED' if i == 0 else ('💰 BUDGET OPTION' if i == 1 else '')
    print(f'{m:<22} {s:>8.1f} {rec}')

## 4. Full Selection Engine — Automated End-to-End

In [ ]:
def select_model(req: TaskRequirements, verbose: bool = True) -> dict:
    """Full automated model selection pipeline."""
    # Step 1: Hard constraint filter
    candidates = filter_models(req) if verbose else [
        n for n, s in MODEL_REGISTRY.items()
        if all(c(req, s) for _, c in HARD_CONSTRAINTS)
    ]
    if not candidates:
        return {'error': 'No models pass hard constraints', 'recommended': None}
    
    # Step 2: Task-fit score
    ranked = score_candidates(req.task_category, candidates)
    
    # Step 3: Calculate monthly costs for top 3
    IN_T, OUT_T = 2000, 400
    cost_data = {}
    for model, _ in ranked[:3]:
        spec = MODEL_REGISTRY[model]
        per_call = IN_T * spec['input_per_1m'] / 1e6 + OUT_T * spec['output_per_1m'] / 1e6
        cost_data[model] = {'per_call': per_call, 'monthly': per_call * req.calls_per_day * 30}
    
    best, budget = ranked[0][0], ranked[1][0] if len(ranked) > 1 else ranked[0][0]
    
    return {
        'recommended': best,
        'budget_option': budget,
        'top_3_ranked': [(m, round(s, 1)) for m, s in ranked[:3]],
        'monthly_costs': {m: f"${c['monthly']:.0f}" for m, c in cost_data.items()},
        'constraints_evaluated': len(MODEL_REGISTRY),
        'candidates_after_filter': len(candidates)
    }

# Test multiple scenarios
SCENARIOS = [
    ('Support chatbot',     TaskRequirements('chat',          'moderate', max_latency_ms=3000, max_cost_per_call=0.005, calls_per_day=10000)),
    ('Doc extractor',       TaskRequirements('extraction',    'simple',   max_cost_per_call=0.001, calls_per_day=50000)),
    ('Code reviewer',       TaskRequirements('coding',        'complex',  min_quality_score=0.90, calls_per_day=500)),
    ('Private on-prem',     TaskRequirements('qa',            'moderate', privacy_required=True, calls_per_day=1000)),
]

for name, req in SCENARIOS:
    result = select_model(req, verbose=False)
    print(f"\n{'─'*55}")
    print(f"Scenario: {name}")
    print(f"  Recommended: {result['recommended']}")
    print(f"  Budget alt:  {result['budget_option']}")
    if result['recommended'] and result['recommended'] in result['monthly_costs']:
        print(f"  Monthly cost (recommended): {result['monthly_costs'][result['recommended']]}")

## 5. PoC Validation — Test on Real Data

In [ ]:
# Simplified PoC: run test cases on selected model, measure pass rate
TEST_CASES = [
    {'prompt': 'Classify as POSITIVE/NEGATIVE/NEUTRAL: "I love this product!"',
     'pass_fn': lambda r: any(w in r.upper() for w in ['POSITIVE','NEGATIVE','NEUTRAL'])},
    {'prompt': 'Classify as POSITIVE/NEGATIVE/NEUTRAL: "This is okay, nothing special."',
     'pass_fn': lambda r: any(w in r.upper() for w in ['POSITIVE','NEGATIVE','NEUTRAL'])},
    {'prompt': 'Classify as POSITIVE/NEGATIVE/NEUTRAL: "Total waste of money."',
     'pass_fn': lambda r: any(w in r.upper() for w in ['POSITIVE','NEGATIVE','NEUTRAL'])},
]

MODEL_TO_TEST = 'gpt-4o-mini'
passed, total_latency = 0, 0
print(f'PoC evaluation: {MODEL_TO_TEST} on {len(TEST_CASES)} test cases\n')

for i, tc in enumerate(TEST_CASES, 1):
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL_TO_TEST,
        messages=[{'role': 'user', 'content': tc['prompt']}],
        max_tokens=10, temperature=0
    )
    latency = (time.perf_counter() - t0) * 1000
    output = r.choices[0].message.content.strip()
    ok = tc['pass_fn'](output)
    passed += int(ok)
    total_latency += latency
    print(f"Case {i}: {'✅' if ok else '❌'} ({latency:.0f}ms) | Output: {output}")

pass_rate = passed / len(TEST_CASES) * 100
avg_latency = total_latency / len(TEST_CASES)
print(f"\nPass rate:   {pass_rate:.0f}%  (target: ≥85%)")
print(f"Avg latency: {avg_latency:.0f}ms")
print(f"Decision:    {'✅ APPROVED' if pass_rate >= 85 else '❌ NEEDS IMPROVEMENT'}")

## 6. Module Summary — All 7 Topics in One Decision Grid

| Topic | Core Tool | Key Decision |
|---|---|---|
| 01 Model Landscape | MODEL_SPECS dict | What specs does each model have? |
| 02 Capability Comparison | Benchmark tests | Which model excels at my task type? |
| 03 Cost & Latency | Cost calculator | What's my monthly budget exposure? |
| 04 Task-Based Selection | TASK_WEIGHTS scoring | Match task type → model strengths |
| 05 Provider APIs | NormalizedResponse | Which SDK SDK patterns to use? |
| 06 Open Source | Ollama / Groq | When does self-hosted make sense? |
| 07 Selection Framework | Full pipeline | Systematic, repeatable decision process |

**The golden rule**: Define requirements → apply hard constraints → score by task fit → validate on real data → monitor and re-evaluate.